# Landmine detection: does CBAM attention help? (training notebook)

This notebook trains the models for the controlled CBAM study on **MineGuard-Thermal v15**:

| Exp. | Question | Runs |
|---|---|---|
| **E1** | Does CBAM help YOLOv5x when both start from the *same* COCO weights? | YOLOv5x vs YOLOv5x+CBAM, original split, seeds 0-2 |
| **E2** | Does it help a small model more? | YOLOv5s vs YOLOv5s+CBAM, original split, seeds 0-2 |
| **E3** | How much do scores drop on *unseen recording sequences*? | YOLOv5x, YOLOv5x+CBAM, YOLO11m on the scene-disjoint split |

E4 (latency + aggregated tables) is in `02_evaluate_kaggle.ipynb`.

**Sessions** (each fits one Kaggle *GPU T4 x2* session; two runs train side by side):

| Session | Runs | Approx. time |
|---|---|---|
| A | YOLOv5x, YOLOv5x+CBAM on the original split, seeds 0 and 1 | ~8.5 h |
| B | YOLOv5x, YOLOv5x+CBAM seed 2; YOLOv5s, YOLOv5s+CBAM seeds 0 and 1 | ~8.5 h |
| C | Scene split: YOLOv5x, YOLOv5x+CBAM, YOLO11m; YOLOv5s, YOLOv5s+CBAM seed 2 | ~8.5 h |
| D (optional) | Scene split: YOLOv5s, YOLOv5s+CBAM, seeds 0-2 | ~6 h |

### How to run on Kaggle
1. **Settings:** *Accelerator* = **GPU T4 x2**, *Internet* = **on**.
2. **Data:** *Add Input* → your private dataset containing the MineGuard-Thermal v15 YOLO export
   (upload `MineGuard-Thermal.v15i.yolov8.zip` once as a Kaggle dataset). Alternative: add a Kaggle
   *Secret* named `ROBOFLOW_API_KEY` and the notebook downloads it from Roboflow.
3. **Smoke test first:** keep `SMOKE_TEST = True` and click *Run All* (about 15-20 min). The last cell
   must show every run as `done`.
4. **Real run:** set `SMOKE_TEST = False` and `SESSION = "A"`, then *Save Version* → **Save & Run All (Commit)**.
   It keeps running after you close the browser.
5. When the version finishes: open it → *Output* → download `outputs_session_A` (or attach the version as
   an input to the evaluation notebook). Then repeat with sessions B and C.

What is fair about the comparison: same pinned YOLOv5 code, same data, hyper-parameters, image size,
epochs and seeds; the CBAM models start from the **same COCO weights** as their baselines (remapped to the
new layer numbering), so the only difference is the 8 CBAM blocks.

In [ ]:
# ============================ CONFIGURATION - edit this cell only ============================
SESSION = "A"        # "A", "B", "C" (required), "D" (optional) or "CUSTOM" - see the table above
CUSTOM_RUNS = []     # used when SESSION = "CUSTOM", e.g. [("yolov5x_cbam", "random", 2), ("yolov5s", "scene", 1)]
SMOKE_TEST = True    # True: 2 epochs on tiny subsets of every split (~15-20 min) to check the pipeline first

EPOCHS, IMGSZ, BATCH = 200, 416, 16   # identical to the 2025 thesis runs
TIME_BUDGET_H = 11.0                  # Kaggle kills sessions at 12 h; runs that would not finish are skipped

YOLOV5_COMMIT = "f9578796b8a64889aa9e66dc06914d19f40d338e"   # ultralytics/yolov5 master, pinned for all runs
ULTRALYTICS_VERSION = "8.4.175"
WORK_DIR = "/tmp/lm"                                          # code, YOLOv5 checkout, split links (not saved)
OUT_DIR = "/kaggle/working/" + ("smoke_outputs" if SMOKE_TEST else f"outputs_session_{SESSION}")
# ==============================================================================================
SRC_DIR, YOLO_DIR, GEN_DIR = f"{WORK_DIR}/src", f"{WORK_DIR}/yolov5", f"{WORK_DIR}/gen"

### Setup (no need to edit below)

In [ ]:
import os, sys, json, platform, subprocess, time
import torch

os.makedirs(SRC_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)
os.environ["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"  # YOLOv5/Ultralytics checkpoints are full pickles


def sh(cmd, cwd=None):
    print("$", cmd)
    subprocess.run(cmd, shell=True, check=True, cwd=cwd)


if not os.path.isdir(YOLO_DIR):
    sh(f"git clone -q https://github.com/ultralytics/yolov5 {YOLO_DIR}")
sh(f"git checkout -q {YOLOV5_COMMIT}", cwd=YOLO_DIR)
sh("pip install -q -r requirements.txt", cwd=YOLO_DIR)
sh(f"pip install -q ultralytics=={ULTRALYTICS_VERSION}")

GPUS = list(range(torch.cuda.device_count())) or ["cpu"]
print("GPUs:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())] or "none (CPU only!)")
if GPUS == ["cpu"] and not SMOKE_TEST:
    raise SystemExit("No GPU found: set Accelerator to 'GPU T4 x2' in the notebook settings.")

Experiment code (from `training/src` in the GitHub repo):

In [ ]:
%%writefile /tmp/lm/src/splits.py
"""Dataset discovery and split construction for MineGuard-Thermal v15 (YOLO format).

Three splits are built from the same 3,042 images:

* ``random`` - the original Roboflow train/valid/test split used in the thesis.
* ``scene``  - whole recording sequences are held out, so no test sequence is seen in training.
* ``*_smoke`` - tiny subsets of the above, used to check the pipeline end to end in minutes.

Split folders contain links to the original files, so building them is cheap.
"""

import csv
import os
import random
import re
import shutil
from collections import Counter, defaultdict
from pathlib import Path

EXPECTED_COUNTS = {"train": 2132, "valid": 607, "test": 303}
IMG_EXTS = {".jpg", ".jpeg", ".png"}
SPLITS = ("train", "valid", "test")

# Scene-disjoint split (experiment E3): these sequences are held out entirely.
# Every other sequence (illia_1/2, MYKHAILO_1/2, DM-11, FlightAltitude) goes to train.
SCENE_HOLDOUT = {
    "test": ("illia_4", "MYKHAILO_4"),
    "valid": ("illia_3", "MYKHAILO_3"),
}


def sequence_of(filename):
    """Map a Roboflow filename to the recording sequence it came from (None if unknown)."""
    stem = filename.split(".rf.")[0]
    if stem.startswith("FlightAltitude"):
        return "FlightAltitude"
    m = re.match(r"(MYKHAILO_\d+|illia_\d+|DM-11)_", stem)
    return m.group(1) if m else None


def find_dataset(search_roots):
    """Return the first directory under ``search_roots`` that holds a YOLO export with train/valid/test."""
    for root in search_roots:
        if not Path(root).exists():
            continue
        for dirpath, dirnames, _ in os.walk(root):
            if set(SPLITS) <= set(dirnames) and Path(dirpath, "train", "images").is_dir():
                return Path(dirpath)
    return None


def download_roboflow(dest, api_key):
    """Fallback when the dataset is not attached as a Kaggle input."""
    from roboflow import Roboflow

    project = Roboflow(api_key=api_key).workspace("dronecv").project("mineguard-thermal")
    ds = project.version(15).download("yolov5", location=str(dest), overwrite=False)
    return Path(ds.location)


def collect_items(src_root):
    """List (original_split, image_path, label_path_or_None, n_boxes) for every image."""
    items = []
    for sp in SPLITS:
        for img in sorted((src_root / sp / "images").iterdir()):
            if img.suffix.lower() not in IMG_EXTS:
                continue
            lab = src_root / sp / "labels" / (img.stem + ".txt")
            n = sum(1 for line in lab.read_text().splitlines() if line.strip()) if lab.exists() else 0
            items.append((sp, img, lab if lab.exists() else None, n))
    return items


def _link(src, dst):
    """Symlink if possible (Kaggle/Linux), otherwise copy (e.g. Windows without dev mode)."""
    if dst.exists():
        return
    dst.parent.mkdir(parents=True, exist_ok=True)
    try:
        os.symlink(src, dst)
    except OSError:
        shutil.copy2(src, dst)


def _write_split(assignment, out_dir):
    """Materialise {split: [items]} under ``out_dir`` and write a data.yaml for YOLOv5/Ultralytics."""
    if out_dir.exists():
        shutil.rmtree(out_dir)
    for sp in SPLITS:
        (out_dir / sp / "images").mkdir(parents=True, exist_ok=True)
        (out_dir / sp / "labels").mkdir(parents=True, exist_ok=True)
        for _, img, lab, _ in assignment[sp]:
            _link(img, out_dir / sp / "images" / img.name)
            if lab is not None:
                _link(lab, out_dir / sp / "labels" / lab.name)
    yaml_path = out_dir / "data.yaml"
    yaml_path.write_text(
        f"path: {out_dir.as_posix()}\n"
        "train: train/images\nval: valid/images\ntest: test/images\n"
        "nc: 1\nnames: ['landmine']\n"
    )
    return yaml_path


def _summarise(name, assignment):
    print(f"\n[{name}] images / boxes per split and sequence")
    seqs = sorted({sequence_of(i[1].name) or "?" for sp in SPLITS for i in assignment[sp]})
    print(f"  {'sequence':16s}" + "".join(f"{sp:>16s}" for sp in SPLITS))
    for seq in seqs:
        row = ""
        for sp in SPLITS:
            sel = [i for i in assignment[sp] if (sequence_of(i[1].name) or "?") == seq]
            row += f"{len(sel):>9d} / {sum(i[3] for i in sel):<5d}"
        print(f"  {seq:16s}{row}")
    print(f"  {'TOTAL':16s}" + "".join(
        f"{len(assignment[sp]):>9d} / {sum(i[3] for i in assignment[sp]):<5d}" for sp in SPLITS))


def build_splits(src_root, out_root, smoke=False, smoke_sizes=(64, 32, 32), seed=0, manifest_path=None):
    """Build the random and scene splits (and smoke subsets). Returns {split_name: data.yaml path}."""
    items = collect_items(src_root)
    counts = Counter(i[0] for i in items)
    if dict(counts) != EXPECTED_COUNTS:
        print(f"WARNING: expected {EXPECTED_COUNTS}, found {dict(counts)} - is this MineGuard-Thermal v15?")

    random_split = {sp: [i for i in items if i[0] == sp] for sp in SPLITS}

    holdout = {seq: sp for sp, seqs in SCENE_HOLDOUT.items() for seq in seqs}
    scene_split = defaultdict(list)
    unknown = []
    for it in items:
        seq = sequence_of(it[1].name)
        if seq is None:
            unknown.append(it[1].name)
        scene_split[holdout.get(seq, "train")].append(it)
    if unknown:
        print(f"WARNING: {len(unknown)} images with unknown sequence were put in train, e.g. {unknown[:3]}")
    leaked = {sequence_of(i[1].name) for i in scene_split["train"]} & set(holdout)
    assert not leaked, f"held-out sequences leaked into train: {leaked}"

    splits = {"random": random_split, "scene": dict(scene_split)}
    if smoke:
        rng = random.Random(seed)
        splits = {
            f"{name}_smoke": {sp: rng.sample(s[sp], min(n, len(s[sp]))) for sp, n in zip(SPLITS, smoke_sizes)}
            for name, s in splits.items()
        }

    yamls = {}
    for name, assignment in splits.items():
        _summarise(name, assignment)
        yamls[name] = _write_split(assignment, Path(out_root) / name)

    if manifest_path is not None:
        Path(manifest_path).parent.mkdir(parents=True, exist_ok=True)
        scene_of = {i[1].name: sp for sp in SPLITS for i in scene_split[sp]}
        with open(manifest_path, "w", newline="") as f:
            w = csv.writer(f)
            w.writerow(["image", "sequence", "boxes", "random_split", "scene_split"])
            for sp, img, _, n in items:
                w.writerow([img.name, sequence_of(img.name) or "", n, sp, scene_of[img.name]])
    return yamls

In [ ]:
%%writefile /tmp/lm/src/setup_yolov5.py
"""Add CBAM to a pinned Ultralytics YOLOv5 checkout and prepare fair, COCO-pretrained CBAM models.

What this does (each step is idempotent):

1. ``patch``     - appends the CBAM module to ``models/common.py`` and teaches ``parse_model`` about it.
2. ``make_yaml`` - writes ``yolov5{s,x}_cbam.yaml``: the official architecture with one CBAM after
                   *every* C3 block in the backbone and the neck (8 CBAMs), all layer references remapped.
3. ``init_ckpt`` - remaps the official COCO ``yolov5{s,x}.pt`` weights into the CBAM layer numbering, so
                   the CBAM model starts from exactly the same pretrained weights as the baseline. Only
                   the CBAM blocks (and, as for the baseline, the 1-class Detect convs) start from scratch.
                   Without this remap YOLOv5 matches weights by layer index and silently drops most of them.
4. ``info``      - parameter / GFLOP table for the four models.

Run as a script inside any Python that can import the YOLOv5 repo, e.g.
``python setup_yolov5.py all --yolo-dir /tmp/lm/yolov5 --gen-dir /tmp/lm/gen --seeds 0 1 2``.
"""

import argparse
import copy
import json
import os
import sys
from pathlib import Path

import yaml

CBAM_SOURCE = '''

# ---------------------------------------------------------------------------------------------
# CBAM: Convolutional Block Attention Module (Woo et al., ECCV 2018), added for the landmine study.
# Pooling uses mean/amax rather than AdaptiveAvg/MaxPool2d because YOLOv5 trains with
# torch.use_deterministic_algorithms(True) and adaptive pooling has no deterministic CUDA backward.
class ChannelAttention(nn.Module):
    """Channel attention: shared MLP over global average- and max-pooled descriptors."""

    def __init__(self, c, r=16):
        super().__init__()
        h = max(c // r, 1)
        self.mlp = nn.Sequential(nn.Conv2d(c, h, 1, bias=False), nn.ReLU(), nn.Conv2d(h, c, 1, bias=False))

    def forward(self, x):
        return torch.sigmoid(self.mlp(x.mean((2, 3), keepdim=True)) + self.mlp(x.amax((2, 3), keepdim=True)))


class SpatialAttention(nn.Module):
    """Spatial attention: 7x7 conv over channel-wise average and max maps."""

    def __init__(self, k=7):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, k, padding=k // 2, bias=False)

    def forward(self, x):
        return torch.sigmoid(self.conv(torch.cat((x.mean(1, keepdim=True), x.amax(1, keepdim=True)), 1)))


class CBAM(nn.Module):
    """Channel attention followed by spatial attention; output has the same shape as the input."""

    def __init__(self, c1, r=16, k=7):
        super().__init__()
        self.ca = ChannelAttention(c1, r)
        self.sa = SpatialAttention(k)

    def forward(self, x):
        x = x * self.ca(x)
        return x * self.sa(x)
'''

PARSE_ANCHOR = "        elif m is nn.BatchNorm2d:\n"
PARSE_BRANCH = "        elif m is CBAM:\n            c2 = ch[f]\n            args = [c2, *args]\n"
IMPORT_ANCHOR = "from models.experimental import MixConv2d\n"
IMPORT_LINE = "from models.common import CBAM\n"


def patch(yolo_dir):
    """Add CBAM to models/common.py and models/yolo.py (safe to call repeatedly)."""
    common = Path(yolo_dir, "models", "common.py")
    text = common.read_text(encoding="utf-8")
    if "class CBAM(" not in text:
        common.write_text(text + CBAM_SOURCE, encoding="utf-8")

    yolo = Path(yolo_dir, "models", "yolo.py")
    text = yolo.read_text(encoding="utf-8")
    if IMPORT_LINE not in text:
        assert text.count(IMPORT_ANCHOR) == 1 and text.count(PARSE_ANCHOR) == 1, (
            "models/yolo.py does not look like the pinned YOLOv5 commit; refusing to patch")
        text = text.replace(IMPORT_ANCHOR, IMPORT_LINE + IMPORT_ANCHOR, 1)
        text = text.replace(PARSE_ANCHOR, PARSE_BRANCH + PARSE_ANCHOR, 1)
        yolo.write_text(text, encoding="utf-8")
    print("patched YOLOv5 with CBAM")


def make_yaml(yolo_dir, variant, gen_dir):
    """Write yolov5{variant}_cbam.yaml; return (path, module_index_map, cbam_indices)."""
    src = yaml.safe_load(Path(yolo_dir, "models", f"yolov5{variant}.yaml").read_text(encoding="utf-8"))
    layers = src["backbone"] + src["head"]
    n_backbone = len(src["backbone"])

    new, module_at, output_at, cbam_idx = [], {}, {}, []
    new_backbone_len = None
    for i, layer in enumerate(layers):
        if i == n_backbone:
            new_backbone_len = len(new)
        module_at[i] = output_at[i] = len(new)
        new.append(copy.deepcopy(layer))
        if layer[2] == "C3":
            cbam_idx.append(len(new))
            output_at[i] = len(new)  # anything that consumed this C3 now consumes its CBAM
            new.append([-1, 1, "CBAM", []])

    def remap(f):
        if isinstance(f, list):
            return [remap(x) for x in f]
        return f if f < 0 else output_at[f]

    for layer in new:
        layer[0] = remap(layer[0])

    out = {k: v for k, v in src.items() if k not in ("backbone", "head")}
    out["backbone"], out["head"] = new[:new_backbone_len], new[new_backbone_len:]
    path = Path(gen_dir, f"yolov5{variant}_cbam.yaml")
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(f"# yolov5{variant}.yaml with CBAM after every C3 (generated by setup_yolov5.py)\n"
                    + yaml.safe_dump(out, sort_keys=False, default_flow_style=None), encoding="utf-8")
    meta = {"module_map": module_at, "cbam_indices": cbam_idx}
    Path(gen_dir, f"yolov5{variant}_cbam.map.json").write_text(json.dumps(meta, indent=1))
    return path, module_at, cbam_idx


def _import_yolov5(yolo_dir):
    sys.path.insert(0, str(Path(yolo_dir).resolve()))
    os.chdir(yolo_dir)  # YOLOv5 resolves some paths relative to the repo root


def init_ckpt(yolo_dir, variant, seed, gen_dir):
    """Create gen_dir/yolov5{variant}_cbam_init_s{seed}.pt from the official COCO weights."""
    out_path = Path(gen_dir, f"yolov5{variant}_cbam_init_s{seed}.pt").resolve()
    if out_path.exists():
        return out_path
    cfg, module_at, cbam_idx = make_yaml(yolo_dir, variant, gen_dir)
    cfg = cfg.resolve()
    _import_yolov5(yolo_dir)
    import torch
    from models.yolo import Model
    from ultralytics.utils.patches import torch_load
    from utils.downloads import attempt_download

    ckpt = torch_load(attempt_download(f"yolov5{variant}.pt"), map_location="cpu")
    coco = ckpt["model"].float().state_dict()

    remapped = {}
    for k, v in coco.items():
        parts = k.split(".")
        parts[1] = str(module_at[int(parts[1])])
        remapped[".".join(parts)] = v

    torch.manual_seed(seed)  # CBAM blocks get a seed-dependent random init, like any new layer
    model = Model(str(cfg), ch=3, nc=80)
    missing, unexpected = model.load_state_dict(remapped, strict=False)
    assert not unexpected, f"unexpected keys: {unexpected[:5]}"
    bad = [k for k in missing if int(k.split(".")[1]) not in cbam_idx]
    assert not bad, f"non-CBAM weights missing after remap: {bad[:5]}"
    print(f"yolov5{variant}_cbam seed {seed}: {len(remapped)} COCO tensors transferred, "
          f"{len(missing)} CBAM tensors initialised from scratch")

    torch.save({"model": copy.deepcopy(model).half(), "epoch": -1, "best_fitness": None, "optimizer": None,
                "ema": None, "updates": None, "opt": None,
                "note": f"COCO yolov5{variant}.pt remapped into the CBAM layout (seed {seed})"}, out_path)
    return out_path


def info(yolo_dir, gen_dir, imgsz=416):
    """Print and return params/GFLOPs for the baseline and CBAM models (nc=1)."""
    _import_yolov5(yolo_dir)
    import thop
    import torch
    from models.common import CBAM
    from models.yolo import Model

    rows = {}
    for variant in ("s", "x"):
        for name, cfg in ((f"yolov5{variant}", Path(yolo_dir, "models", f"yolov5{variant}.yaml")),
                          (f"yolov5{variant}_cbam", Path(gen_dir, f"yolov5{variant}_cbam.yaml").resolve())):
            if not cfg.exists():
                continue
            m = Model(str(cfg), ch=3, nc=1).eval()
            params = sum(p.numel() for p in m.parameters())
            macs = thop.profile(copy.deepcopy(m), inputs=(torch.zeros(1, 3, imgsz, imgsz),), verbose=False)[0]
            n_cbam = sum(isinstance(x, CBAM) for x in m.modules())
            rows[name] = {"params": params, "gflops": 2 * macs / 1e9, "cbam_blocks": n_cbam}
    print(f"\n{'model':16s}{'params':>14s}{'GFLOPs@' + str(imgsz):>14s}{'CBAM blocks':>13s}")
    for k, r in rows.items():
        print(f"{k:16s}{r['params']:>14,d}{r['gflops']:>14.1f}{r['cbam_blocks']:>13d}")
    for v in ("s", "x"):
        if f"yolov5{v}_cbam" in rows:
            assert rows[f"yolov5{v}_cbam"]["cbam_blocks"] == 8, "expected one CBAM after each of the 8 C3 blocks"
    return rows


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("step", choices=["patch", "yaml", "init", "info", "all"])
    ap.add_argument("--yolo-dir", required=True)
    ap.add_argument("--gen-dir", required=True)
    ap.add_argument("--variants", nargs="+", default=["s", "x"])
    ap.add_argument("--seeds", nargs="+", type=int, default=[0])
    ap.add_argument("--imgsz", type=int, default=416)
    a = ap.parse_args()
    a.gen_dir = str(Path(a.gen_dir).resolve())
    a.yolo_dir = str(Path(a.yolo_dir).resolve())
    if a.step in ("patch", "all"):
        patch(a.yolo_dir)
    if a.step in ("yaml", "all"):
        for v in a.variants:
            print("wrote", make_yaml(a.yolo_dir, v, a.gen_dir)[0])
    if a.step in ("init", "all"):
        here = os.getcwd()
        for v in a.variants:
            for s in a.seeds:
                init_ckpt(a.yolo_dir, v, s, a.gen_dir)
                os.chdir(here)
    if a.step in ("info", "all"):
        res = info(a.yolo_dir, a.gen_dir, a.imgsz)
        Path(a.gen_dir, "model_info.json").write_text(json.dumps(res, indent=2))

In [ ]:
%%writefile /tmp/lm/src/runner.py
"""Run matrix and a small multi-GPU scheduler for the attention study.

Every run is (model, split, seed). Runs are queued and each GPU takes the next run that still fits in
the session's time budget, so one Kaggle "T4 x2" session trains two models side by side.
Each run writes to OUT/<model>__<split>__s<seed>/ : weights/best.pt, results.csv, plots, train.log,
metrics.json (val + test) and run.json (command, timing, status).
"""

import json
import os
import re
import shutil
import subprocess
import sys
import threading
import time
from dataclasses import dataclass
from pathlib import Path

# est_h: rough hours per 200-epoch run on one T4 while a second run uses the other GPU.
MODELS = {
    "yolov5s":      {"kind": "yolov5", "variant": "s", "cbam": False, "est_h": 1.8},
    "yolov5s_cbam": {"kind": "yolov5", "variant": "s", "cbam": True,  "est_h": 2.0},
    "yolov5x":      {"kind": "yolov5", "variant": "x", "cbam": False, "est_h": 4.0},
    "yolov5x_cbam": {"kind": "yolov5", "variant": "x", "cbam": True,  "est_h": 4.5},
    "yolo11m":      {"kind": "ultralytics", "weights": "yolo11m.pt",   "est_h": 3.5},
}

# Sessions sized for one Kaggle "GPU T4 x2" session (12 h hard limit).
SESSIONS = {
    # E1: YOLOv5x vs YOLOv5x+CBAM on the original split, seeds 0 and 1           (~8.5 h)
    "A": [("yolov5x", "random", 0), ("yolov5x_cbam", "random", 0),
          ("yolov5x", "random", 1), ("yolov5x_cbam", "random", 1)],
    # E1 seed 2 + E2: YOLOv5s vs YOLOv5s+CBAM, seeds 0 and 1                      (~8.5 h)
    "B": [("yolov5x", "random", 2), ("yolov5x_cbam", "random", 2),
          ("yolov5s", "random", 0), ("yolov5s_cbam", "random", 0),
          ("yolov5s", "random", 1), ("yolov5s_cbam", "random", 1)],
    # E3: scene-disjoint split + E2 seed 2                                         (~8.5 h)
    "C": [("yolov5x", "scene", 0), ("yolov5x_cbam", "scene", 0), ("yolo11m", "scene", 0),
          ("yolov5s", "random", 2), ("yolov5s_cbam", "random", 2)],
    # Optional: E3 with the small models over 3 seeds                             (~6 h)
    "D": [(m, "scene", s) for s in (0, 1, 2) for m in ("yolov5s", "yolov5s_cbam")],
}


@dataclass
class Run:
    model: str
    split: str
    seed: int

    @property
    def name(self):
        return f"{self.model}__{self.split}__s{self.seed}"


def build_command(run, cfg, data_yaml, run_dir, device):
    """Return (argv, cwd) for one training run."""
    m = MODELS[run.model]
    if m["kind"] == "ultralytics":
        argv = [sys.executable, str(Path(cfg["src_dir"], "train_ultralytics.py")), "--model", m["weights"],
                "--data", str(data_yaml), "--project", str(run_dir.parent), "--name", run_dir.name,
                "--seed", str(run.seed), "--epochs", str(cfg["epochs"]), "--imgsz", str(cfg["imgsz"]),
                "--batch", str(cfg["batch"]), "--workers", str(cfg["workers"]), "--device", device]
        return argv, cfg["work_dir"]
    v = m["variant"]
    if m["cbam"]:
        model_cfg = Path(cfg["gen_dir"], f"yolov5{v}_cbam.yaml")
        weights = Path(cfg["gen_dir"], f"yolov5{v}_cbam_init_s{run.seed}.pt")
    else:
        model_cfg = Path("models", f"yolov5{v}.yaml")
        weights = f"yolov5{v}.pt"
    argv = [sys.executable, "train.py", "--img", str(cfg["imgsz"]), "--batch", str(cfg["batch"]),
            "--epochs", str(cfg["epochs"]), "--data", str(data_yaml), "--cfg", str(model_cfg),
            "--weights", str(weights), "--seed", str(run.seed), "--cache", "ram", "--workers", str(cfg["workers"]),
            "--patience", "100", "--device", device, "--project", str(run_dir.parent), "--name", run_dir.name,
            "--exist-ok"]
    return argv, cfg["yolo_dir"]


def _env_for(gpu):
    env = dict(os.environ)
    env["PYTHONUNBUFFERED"] = "1"
    env.setdefault("TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD", "1")
    if gpu != "cpu":
        env["CUDA_VISIBLE_DEVICES"] = str(gpu)
    return env


def execute(run, cfg, data_yaml, gpu):
    """Train + evaluate one run; never raises (failures are recorded in run.json)."""
    run_dir = Path(cfg["out_dir"], run.name)
    run_dir.mkdir(parents=True, exist_ok=True)
    device = "cpu" if gpu == "cpu" else "0"
    meta = {"run": run.name, "model": run.model, "split": run.split, "seed": run.seed, "gpu": gpu,
            "epochs": cfg["epochs"], "imgsz": cfg["imgsz"], "batch": cfg["batch"], "data": str(data_yaml),
            "status": "running", "started": time.strftime("%Y-%m-%d %H:%M:%S")}
    t0 = time.time()
    argv, cwd = build_command(run, cfg, data_yaml, run_dir, device)
    meta["command"] = " ".join(argv)
    (run_dir / "run.json").write_text(json.dumps(meta, indent=2))
    log_path = run_dir / "train.log"
    try:
        with open(log_path, "w", encoding="utf-8") as log:
            rc = subprocess.run(argv, cwd=cwd, env=_env_for(gpu), stdout=log, stderr=subprocess.STDOUT).returncode
        if rc != 0:
            raise RuntimeError(f"training exited with code {rc}")
        meta["train_hours"] = round((time.time() - t0) / 3600, 3)
        log_text = log_path.read_text(encoding="utf-8", errors="replace")
        hit = re.findall(r"Transferred (\d+)/(\d+) items", log_text)
        if hit:
            meta["transferred_items"] = f"{hit[-1][0]}/{hit[-1][1]}"
        if MODELS[run.model]["kind"] == "yolov5":
            best = run_dir / "weights" / "best.pt"
            eval_argv = [sys.executable, str(Path(cfg["src_dir"], "eval_yolov5.py")), "--yolo-dir", cfg["yolo_dir"],
                         "--weights", str(best), "--data", str(data_yaml), "--out", str(run_dir),
                         "--imgsz", str(cfg["imgsz"]), "--workers", str(cfg["workers"])]
            with open(run_dir / "eval.log", "w", encoding="utf-8") as log:
                rc = subprocess.run(eval_argv, cwd=cfg["yolo_dir"], env=_env_for(gpu), stdout=log,
                                    stderr=subprocess.STDOUT).returncode
            if rc != 0:
                raise RuntimeError(f"evaluation exited with code {rc}")
        last = run_dir / "weights" / "last.pt"
        if last.exists():
            last.unlink()  # best.pt is all we need; keeps the Kaggle output small
        meta["status"] = "done"
        meta["test"] = json.loads((run_dir / "metrics.json").read_text())["test"]
    except Exception as e:  # keep the session going; the log has the details
        meta["status"] = "failed"
        meta["error"] = str(e)
        tail = log_path.read_text(encoding="utf-8", errors="replace").splitlines()[-25:] if log_path.exists() else []
        print(f"\n!!! {run.name} FAILED: {e}\n" + "\n".join(tail))
    meta["total_hours"] = round((time.time() - t0) / 3600, 3)
    (run_dir / "run.json").write_text(json.dumps(meta, indent=2))
    return meta


def _progress(run_dir):
    for csv_path in (run_dir / "results.csv",):
        if csv_path.exists():
            lines = [ln for ln in csv_path.read_text().splitlines() if ln.strip()]
            if len(lines) > 1:
                head = [h.strip() for h in lines[0].split(",")]
                last = [x.strip() for x in lines[-1].split(",")]
                key = next((k for k in head if "mAP_0.5:0.95" in k or "mAP50-95" in k), None)
                score = f"{float(last[head.index(key)]):.3f}" if key else "?"
                return f"epoch {len(lines) - 1}, val mAP50-95 {score}"
    return "starting"


def run_session(runs, cfg, data_yamls, gpus, budget_h, poll_s=600):
    """Run ``runs`` across ``gpus``; a GPU only starts a run if its estimate fits in the remaining budget."""
    queue = [Run(*r) for r in runs]
    lock = threading.Lock()
    active, results = {}, {}
    t0 = time.time()

    def worker(gpu):
        while True:
            with lock:
                elapsed = (time.time() - t0) / 3600
                pick = next((r for r in queue if elapsed + MODELS[r.model]["est_h"] <= budget_h), None)
                if pick is None:
                    return
                queue.remove(pick)
                active[gpu] = pick
            print(f"[{time.strftime('%H:%M')}] GPU {gpu}: start {pick.name}")
            results[pick.name] = execute(pick, cfg, data_yamls[pick.split], gpu)
            r = results[pick.name]
            print(f"[{time.strftime('%H:%M')}] GPU {gpu}: {pick.name} {r['status']} "
                  f"in {r['total_hours']:.2f} h" + (f" | test mAP50-95 {r['test']['map50_95']:.4f}"
                                                    if r["status"] == "done" else ""))
            with lock:
                active.pop(gpu, None)

    threads = [threading.Thread(target=worker, args=(g,), daemon=True) for g in gpus]
    for t in threads:
        t.start()
    while any(t.is_alive() for t in threads):
        for t in threads:
            t.join(timeout=poll_s / max(len(threads), 1))
        with lock:
            status = {g: f"{r.name}: {_progress(Path(cfg['out_dir'], r.name))}" for g, r in active.items()}
        if status:
            print(f"[{time.strftime('%H:%M')}] elapsed {(time.time() - t0) / 3600:.2f} h | "
                  + " | ".join(f"GPU {g} {s}" for g, s in status.items()))

    skipped = [r.name for r in queue]
    summary = {"runs": results, "skipped_for_time": skipped, "hours": round((time.time() - t0) / 3600, 2)}
    Path(cfg["out_dir"], "session_summary.json").write_text(json.dumps(summary, indent=2))
    return summary


def print_summary(summary):
    print(f"\n{'run':34s}{'status':>8s}{'hours':>7s}{'P':>8s}{'R':>8s}{'mAP50':>8s}{'mAP50-95':>10s}")
    for name, r in summary["runs"].items():
        t = r.get("test", {})
        print(f"{name:34s}{r['status']:>8s}{r['total_hours']:>7.2f}" + (
            f"{t['precision']:>8.3f}{t['recall']:>8.3f}{t['map50']:>8.3f}{t['map50_95']:>10.3f}" if t else ""))
    if summary["skipped_for_time"]:
        print("\nNot started (would not fit in the time budget) - run them in a later session with CUSTOM_RUNS:")
        for n in summary["skipped_for_time"]:
            print("  ", n)


def copy_split_manifest(src, out_dir):
    if src and Path(src).exists():
        shutil.copy2(src, Path(out_dir, Path(src).name))

In [ ]:
%%writefile /tmp/lm/src/eval_yolov5.py
"""Evaluate a trained YOLOv5 checkpoint on the val and test splits and write metrics.json.

Usage: python eval_yolov5.py --yolo-dir /tmp/lm/yolov5 --weights best.pt --data data.yaml --out run_dir
"""

import argparse
import copy
import json
import os
import sys
from pathlib import Path


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--yolo-dir", required=True)
    ap.add_argument("--weights", required=True)
    ap.add_argument("--data", required=True)
    ap.add_argument("--out", required=True)
    ap.add_argument("--imgsz", type=int, default=416)
    ap.add_argument("--batch", type=int, default=32)
    ap.add_argument("--workers", type=int, default=8)
    a = ap.parse_args()
    weights, data, out = (str(Path(p).resolve()) for p in (a.weights, a.data, a.out))

    sys.path.insert(0, str(Path(a.yolo_dir).resolve()))
    os.chdir(a.yolo_dir)
    import thop
    import torch
    import val
    from models.experimental import attempt_load

    cuda = torch.cuda.is_available()
    res = {"weights": weights, "imgsz": a.imgsz}
    for task in ("val", "test"):
        (mp, mr, map50, map5095, *_), _, t = val.run(
            data=data, weights=weights, imgsz=a.imgsz, batch_size=a.batch, task=task,
            device="0" if cuda else "cpu", half=cuda, workers=a.workers, project=out, name=f"eval_{task}",
            exist_ok=True, plots=True)
        res[task] = {"precision": float(mp), "recall": float(mr), "f1": float(2 * mp * mr / (mp + mr + 1e-16)),
                     "map50": float(map50), "map50_95": float(map5095),
                     "speed_ms_per_img": {"pre": t[0], "inference": t[1], "nms": t[2]}}

    model = attempt_load(weights, device="cpu", fuse=False).float().eval()
    res["params"] = int(sum(p.numel() for p in model.parameters()))
    macs = thop.profile(copy.deepcopy(model), inputs=(torch.zeros(1, 3, a.imgsz, a.imgsz),), verbose=False)[0]
    res["gflops"] = 2 * macs / 1e9
    Path(out, "metrics.json").write_text(json.dumps(res, indent=2))
    print(json.dumps(res, indent=2))


if __name__ == "__main__":  # required on Windows, where data-loader workers re-import this file
    main()

In [ ]:
%%writefile /tmp/lm/src/train_ultralytics.py
"""Train an Ultralytics model (YOLO11) and evaluate it on val and test; writes metrics.json like eval_yolov5.py.

Usage: python train_ultralytics.py --model yolo11m.pt --data data.yaml --project out --name run --seed 0
"""

import argparse
import json
from pathlib import Path

from ultralytics import YOLO
from ultralytics.utils.torch_utils import get_flops, get_num_params


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--model", required=True)
    ap.add_argument("--data", required=True)
    ap.add_argument("--project", required=True)
    ap.add_argument("--name", required=True)
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--epochs", type=int, default=200)
    ap.add_argument("--imgsz", type=int, default=416)
    ap.add_argument("--batch", type=int, default=16)
    ap.add_argument("--workers", type=int, default=2)
    ap.add_argument("--device", default="0")
    a = ap.parse_args()
    project = str(Path(a.project).resolve())

    YOLO(a.model).train(data=a.data, epochs=a.epochs, imgsz=a.imgsz, batch=a.batch, seed=a.seed, deterministic=True,
                        device=a.device, workers=a.workers, cache="ram", patience=100, project=project, name=a.name,
                        exist_ok=True, plots=True)

    run_dir = Path(project, a.name)
    best = run_dir / "weights" / "best.pt"
    model = YOLO(str(best))
    res = {"weights": str(best), "imgsz": a.imgsz}
    for split in ("val", "test"):
        r = model.val(data=a.data, split=split, imgsz=a.imgsz, batch=32, device=a.device, workers=a.workers,
                      project=str(run_dir),
                      name=f"eval_{split}", exist_ok=True, plots=True)
        mp, mr = float(r.box.mp), float(r.box.mr)
        res[split] = {"precision": mp, "recall": mr, "f1": 2 * mp * mr / (mp + mr + 1e-16),
                      "map50": float(r.box.map50), "map50_95": float(r.box.map),
                      "speed_ms_per_img": {k: float(v) for k, v in r.speed.items()}}
    res["params"] = int(get_num_params(model.model))
    res["gflops"] = float(get_flops(model.model, imgsz=a.imgsz))
    (run_dir / "metrics.json").write_text(json.dumps(res, indent=2))
    print(json.dumps(res, indent=2))


if __name__ == "__main__":  # required on Windows, where data-loader workers re-import this file
    main()

In [ ]:
sys.path.insert(0, SRC_DIR)
import runner, splits

RUNS = runner.SESSIONS[SESSION] if SESSION != "CUSTOM" else CUSTOM_RUNS
if SMOKE_TEST:  # every model once on the original split + one scene-split run, 2 epochs each
    RUNS = [(m, "random", 0) for m in runner.MODELS] + [("yolov5s_cbam", "scene", 0)]
    EPOCHS, TIME_BUDGET_H = 2, 99
print(f"{len(RUNS)} runs:", *RUNS, sep="\n  ")

# ---- data -------------------------------------------------------------------------------------
src = splits.find_dataset(["/kaggle/input"])
if src is None:
    from kaggle_secrets import UserSecretsClient
    sh("pip install -q roboflow")
    src = splits.download_roboflow(f"{WORK_DIR}/roboflow", UserSecretsClient().get_secret("ROBOFLOW_API_KEY"))
print("dataset:", src)
yamls = splits.build_splits(src, f"{WORK_DIR}/data", smoke=SMOKE_TEST, manifest_path=f"{OUT_DIR}/split_manifest.csv")
DATA_YAMLS = {k.replace("_smoke", ""): v for k, v in yamls.items()}

In [ ]:
# ---- CBAM patch, CBAM configs, COCO-initialised CBAM checkpoints, model sizes ----------------
v5 = [runner.MODELS[m] for m, _, _ in RUNS if runner.MODELS[m]["kind"] == "yolov5"]
variants = sorted({m["variant"] for m in v5}) or ["s"]
seeds = sorted({s for _, _, s in RUNS})
sh(f"{sys.executable} {SRC_DIR}/setup_yolov5.py all --yolo-dir {YOLO_DIR} --gen-dir {GEN_DIR} "
   f"--variants {' '.join(variants)} --seeds {' '.join(map(str, seeds))} --imgsz {IMGSZ}")

In [ ]:
# ---- train + evaluate -----------------------------------------------------------------------
import ultralytics
env = {"python": platform.python_version(), "torch": torch.__version__, "cuda": torch.version.cuda,
       "gpus": [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],
       "yolov5_commit": YOLOV5_COMMIT, "ultralytics": ultralytics.__version__, "session": SESSION,
       "smoke_test": SMOKE_TEST, "epochs": EPOCHS, "imgsz": IMGSZ, "batch": BATCH, "runs": RUNS,
       "started": time.strftime("%Y-%m-%d %H:%M:%S")}
json.dump(env, open(f"{OUT_DIR}/env.json", "w"), indent=2)
for f in ("model_info.json", "yolov5s_cbam.yaml", "yolov5x_cbam.yaml"):
    if os.path.exists(f"{GEN_DIR}/{f}"):
        sh(f"cp {GEN_DIR}/{f} {OUT_DIR}/")

cfg = {"work_dir": WORK_DIR, "yolo_dir": YOLO_DIR, "gen_dir": GEN_DIR, "src_dir": SRC_DIR, "out_dir": OUT_DIR,
       "epochs": EPOCHS, "imgsz": IMGSZ, "batch": BATCH, "workers": 2 if len(GPUS) > 1 else 4}
summary = runner.run_session(RUNS, cfg, DATA_YAMLS, GPUS, TIME_BUDGET_H, poll_s=120 if SMOKE_TEST else 900)
runner.print_summary(summary)

### After the session
* Every run folder in the output has `weights/best.pt`, `results.csv`, training plots, `train.log`,
  `metrics.json` (val + **test** metrics, params, GFLOPs) and `run.json` (command, timing, status).
* `session_summary.json` lists finished, failed and skipped runs. Re-run anything skipped or failed in a
  later session with `SESSION = "CUSTOM"`.
* For the CBAM runs, `run.json` → `transferred_items` should be **larger** than for the matching baseline
  (the extra items are the CBAM blocks); if it is much smaller, the COCO weights were not transferred.